Dane startowe

In [2]:
import pandas as pd
import numpy as np
# --- Tabela 1: autorzy ---
autorzy = pd.DataFrame({
 'autor_id': [1, 2, 3, 4, 5, 6, 7, 8],
 'imie': ['Olga', 'Stanisław', 'Andrzej', 'Wisława', 'Ryszard',
 'Dorota', 'Szczepan', 'Jacek'],
 'nazwisko': ['Tokarczuk', 'Lem', 'Sapkowski', 'Szymborska', 'Kapuściński',
 'Masłowska', 'Twardoch', 'Dehnel'],
 'kraj': ['Polska', 'Polska', 'Polska', 'Polska', 'Polska',
 'Polska', 'Polska', 'Polska'],
 'nagrody': ['Nobel', 'SFF', 'SFF', 'Nobel', 'Reporter',
 'Polityka', 'NIKE', 'NIKE']
})
# --- Tabela 2: książki ---
ksiazki = pd.DataFrame({
 'ksiazka_id': [101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112],
 'autor_id': [1, 1, 2, 2, 3, 3, 4, 5, 6, 7, 7, 8],
 'tytul': ['Księgi Jakubowe', 'Bieguni', 'Solaris', 'Cyberiada',
 'Wiedźmin', 'Narrenturm', 'Wiersze wybrane', 'Podróże z Herodotem',
 'Wojna polsko-ruska', 'Morfina', 'Król', 'Lala'],
 'kategoria': ['historyczna', 'obyczajowa', 'sci-fi', 'sci-fi',
 'fantasy', 'fantasy', 'poezja', 'reportaz',
 'obyczajowa', 'historyczna', 'historyczna', 'obyczajowa'],
 'cena': [79.90, 49.00, 39.00, 42.00, 45.00, 55.00, 35.00,
 52.00, 38.00, 48.00, 59.00, 44.00],
 'strony': [912, 376, 198, 295, 320, 512, 180, 263, 153, 618, 688, 432]
})
# --- Tabela 3: zamówienia ---
np.random.seed(2026)
n = 80
zamowienia = pd.DataFrame({
 'zam_id': range(5001, 5001 + n),
 'ksiazka_id': np.random.choice(ksiazki['ksiazka_id'], size=n),
 'ilosc': np.random.randint(1, 6, size=n),
 'data': pd.to_datetime('2026-01-01') +
pd.to_timedelta(np.random.randint(0, 120, n), unit='D'),
 'kanal': np.random.choice(['web', 'aplikacja', 'telefon'], size=n, p=
[0.6, 0.3, 0.1]),
 'miasto': np.random.choice(['Warszawa', 'Kraków', 'Wrocław', 'Gdańsk',
'Poznań'], size=n)
})
print(f"Autorzy: {autorzy.shape}")
print(f"Ksiazki: {ksiazki.shape}")
print(f"Zamowienia: {zamowienia.shape}")


Autorzy: (8, 5)
Ksiazki: (12, 6)
Zamowienia: (80, 6)


Ćwiczenie 1

In [4]:
ksiazki_z_autorami = ksiazki.merge(autorzy, on='autor_id')
print(f"Shape: {ksiazki_z_autorami.shape}")
ksiazki_z_autorami.head()

Shape: (12, 10)


,ksiazka_id,autor_id,tytul,kategoria,cena,strony,imie,nazwisko,kraj,nagrody
0,101,1,Księgi Jakubowe,historyczna,79.9,912,Olga,Tokarczuk,Polska,Nobel
1,102,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel
2,103,2,Solaris,sci-fi,39.0,198,Stanisław,Lem,Polska,SFF
3,104,2,Cyberiada,sci-fi,42.0,295,Stanisław,Lem,Polska,SFF
4,105,3,Wiedźmin,fantasy,45.0,320,Andrzej,Sapkowski,Polska,SFF


Zadanie 1a

## Zadanie 1a — Wynik merge

- **Liczba wierszy:** 12 (każda książka dopasowana do swojego autora)
- **Liczba kolumn:** 10 (6 z tabeli ksiazki + 4 nowe z tabeli autorzy)
- **Nowe kolumny z tabeli autorzy:** `imie`, `nazwisko`, `kraj`, `nagrody`

In [7]:
ksiazki_test = pd.concat([
    ksiazki,
    pd.DataFrame({'ksiazka_id': [999], 'autor_id': [999],
                  'tytul': ['Ksiazka bez autora'], 'kategoria': ['test'],
                  'cena': [30.0], 'strony': [100]})
], ignore_index=True)

print(f"ksiazki_test shape: {ksiazki_test.shape}")
ksiazki_test.tail(3)

ksiazki_test shape: (13, 6)


,ksiazka_id,autor_id,tytul,kategoria,cena,strony
10,111,7,Król,historyczna,59.0,688
11,112,8,Lala,obyczajowa,44.0,432
12,999,999,Ksiazka bez autora,test,30.0,100


Zadanie 1b

In [8]:
inner = ksiazki_test.merge(autorzy, on='autor_id', how='inner')
left  = ksiazki_test.merge(autorzy, on='autor_id', how='left')
right = ksiazki_test.merge(autorzy, on='autor_id', how='right')
outer = ksiazki_test.merge(autorzy, on='autor_id', how='outer')

print(f"inner: {inner.shape[0]} wierszy")
print(f"left:  {left.shape[0]} wierszy")
print(f"right: {right.shape[0]} wierszy")
print(f"outer: {outer.shape[0]} wierszy")

audyt = ksiazki_test.merge(autorzy, on='autor_id', how='outer', indicator=True)
print("\nDiagnostyka:")
print(audyt['_merge'].value_counts())

inner: 12 wierszy
left:  13 wierszy
right: 12 wierszy
outer: 13 wierszy

Diagnostyka:
_merge
both          12
left_only      1
right_only     0
Name: count, dtype: int64


Zadanie 1c



`inner` zwraca tylko wiersze z dopasowaniem po obu stronach — książka z `autor_id=999`
nie istnieje w tabeli `autorzy`, więc zostaje pominięta.
`left` zachowuje wszystkie wiersze z lewej tabeli i uzupełnia brakujące dane z prawej wartościami `NaN`.

Ćwiczenie 2

In [9]:
# Połącz: zamowienia + ksiazki + autorzy
pelne = (
 zamowienia
 .merge(ksiazki, on='ksiazka_id')
 .merge(autorzy, on='autor_id')
)
print(f"Pelna tabela: {pelne.shape}")
pelne.head()


Pelna tabela: (80, 15)


,zam_id,ksiazka_id,ilosc,data,kanal,miasto,autor_id,tytul,kategoria,cena,strony,imie,nazwisko,kraj,nagrody
0,5001,102,5,2026-04-05,web,Poznań,1,Bieguni,obyczajowa,49.0,376,Olga,Tokarczuk,Polska,Nobel
1,5002,107,2,2026-02-08,web,Poznań,4,Wiersze wybrane,poezja,35.0,180,Wisława,Szymborska,Polska,Nobel
2,5003,111,1,2026-04-19,web,Poznań,7,Król,historyczna,59.0,688,Szczepan,Twardoch,Polska,NIKE
3,5004,109,4,2026-02-24,web,Warszawa,6,Wojna polsko-ruska,obyczajowa,38.0,153,Dorota,Masłowska,Polska,Polityka
4,5005,105,2,2026-01-10,web,Warszawa,3,Wiedźmin,fantasy,45.0,320,Andrzej,Sapkowski,Polska,SFF


Zadanie 2a

In [10]:
pelne['wartosc'] = pelne['ilosc'] * pelne['cena']
pelne['miesiac'] = pelne['data'].dt.month
pelne['autor_pelne'] = pelne['imie'] + ' ' + pelne['nazwisko']
pelne['kategoria_cenowa'] = np.where(pelne['cena'] >= 50, 'droga', 'tania')

print(f"Kolumny po dodaniu: {pelne.columns.tolist()}")

Kolumny po dodaniu: ['zam_id', 'ksiazka_id', 'ilosc', 'data', 'kanal', 'miasto', 'autor_id', 'tytul', 'kategoria', 'cena', 'strony', 'imie', 'nazwisko', 'kraj', 'nagrody', 'wartosc', 'miesiac', 'autor_pelne', 'kategoria_cenowa']


Zadanie 2b

In [11]:
print(f"Liczba zamówień: {len(pelne)}")
print(f"Łączny przychód: {pelne['wartosc'].sum():.2f} zł")
print(f"Średnia wartość zamówienia: {pelne['wartosc'].mean():.2f} zł")
print(f"Unikalne tytuły: {pelne['tytul'].nunique()}")

Liczba zamówień: 80
Łączny przychód: 11963.80 zł
Średnia wartość zamówienia: 149.55 zł
Unikalne tytuły: 12


Ćwiczenie 3

Zadanie 3a

In [12]:
print("Przychód per kategoria:")
print(pelne.groupby('kategoria')['wartosc'].sum().sort_values(ascending=False).round(2))

Przychód per kategoria:
kategoria
historyczna    4136.8
sci-fi         2334.0
obyczajowa     1999.0
fantasy        1930.0
reportaz       1144.0
poezja          420.0
Name: wartosc, dtype: float64


Zadanie 3b

In [13]:
print("Raport per kategoria:")
print(pelne.groupby('kategoria')['wartosc'].agg(['count', 'mean', 'sum']).round(2))

Raport per kategoria:
             count    mean     sum
kategoria                         
fantasy         14  137.86  1930.0
historyczna     22  188.04  4136.8
obyczajowa      17  117.59  1999.0
poezja           7   60.00   420.0
reportaz         5  228.80  1144.0
sci-fi          15  155.60  2334.0


Zadanie 3c

In [14]:
raport_autor = pelne.groupby('autor_pelne').agg(
    liczba_zamowien=('zam_id', 'count'),
    laczna_sprzedaz=('wartosc', 'sum'),
    sredni_ilosc=('ilosc', 'mean')
).round(2).sort_values('laczna_sprzedaz', ascending=False)
print(raport_autor)

                     liczba_zamowien  laczna_sprzedaz  sredni_ilosc
autor_pelne                                                        
Olga Tokarczuk                    18           3389.8          2.72
Stanisław Lem                     15           2334.0          3.80
Andrzej Sapkowski                 14           1930.0          2.86
Szczepan Twardoch                 11           1580.0          2.91
Ryszard Kapuściński                5           1144.0          4.40
Jacek Dehnel                       5            748.0          3.40
Wisława Szymborska                 7            420.0          1.71
Dorota Masłowska                   5            418.0          2.20


Zadanie 3d

In [15]:
print("Przychód per kategoria × kanał:")
print(pelne.groupby(['kategoria', 'kanal'])['wartosc'].sum().round(2))

Przychód per kategoria × kanał:
kategoria    kanal    
fantasy      aplikacja     475.0
             telefon       225.0
             web          1230.0
historyczna  aplikacja    1988.5
             telefon       527.4
             web          1620.9
obyczajowa   aplikacja     586.0
             telefon       136.0
             web          1277.0
poezja       aplikacja      70.0
             telefon        70.0
             web           280.0
reportaz     aplikacja     416.0
             telefon       260.0
             web           468.0
sci-fi       aplikacja     534.0
             telefon       279.0
             web          1521.0
Name: wartosc, dtype: float64


Zadanie 3e

In [16]:
pelne['wartosc_kat'] = pelne.groupby('kategoria')['wartosc'].transform('sum')
pelne['udzial_w_kategorii_pct'] = (pelne['wartosc'] / pelne['wartosc_kat'] * 100).round(2)

print("Suma udziałów per kategoria (każda powinna = 100%):")
print(pelne.groupby('kategoria')['udzial_w_kategorii_pct'].sum().round(1))

Suma udziałów per kategoria (każda powinna = 100%):
kategoria
fantasy        100.0
historyczna    100.0
obyczajowa     100.0
poezja         100.0
reportaz       100.0
sci-fi         100.0
Name: udzial_w_kategorii_pct, dtype: float64


Ćwiczenie 4

Zadanie 4a

In [17]:
pivot = pd.pivot_table(pelne, index='kategoria', columns='miesiac',
                       values='wartosc', aggfunc='sum', fill_value=0)
print(pivot.round(2))

miesiac           1       2      3      4
kategoria                                
fantasy       595.0   410.0  270.0  655.0
historyczna  1342.5  1790.9  634.4  369.0
obyczajowa    301.0   623.0  274.0  801.0
poezja        105.0    70.0  140.0  105.0
reportaz      416.0     0.0  728.0    0.0
sci-fi        765.0   615.0  672.0  282.0


Zadanie 4b

In [18]:
pivot_razem = pd.pivot_table(pelne, index='kategoria', columns='miesiac',
                              values='wartosc', aggfunc='sum', fill_value=0,
                              margins=True, margins_name='RAZEM')
print(pivot_razem.round(2))

miesiac           1       2       3       4    RAZEM
kategoria                                           
fantasy       595.0   410.0   270.0   655.0   1930.0
historyczna  1342.5  1790.9   634.4   369.0   4136.8
obyczajowa    301.0   623.0   274.0   801.0   1999.0
poezja        105.0    70.0   140.0   105.0    420.0
reportaz      416.0     0.0   728.0     0.0   1144.0
sci-fi        765.0   615.0   672.0   282.0   2334.0
RAZEM        3524.5  3508.9  2718.4  2212.0  11963.8


Zadanie 4c

In [19]:
ct = pd.crosstab(pelne['kanal'], pelne['miasto'])
print(ct)

miasto     Gdańsk  Kraków  Poznań  Warszawa  Wrocław
kanal                                               
aplikacja       7       1       5         6        4
telefon         1       4       1         3        2
web             5       9      14         9        9


Zadanie 4d

In [20]:
ct_pct = pd.crosstab(pelne['kanal'], pelne['miasto'], normalize='index') * 100
print(ct_pct.round(1))

miasto     Gdańsk  Kraków  Poznań  Warszawa  Wrocław
kanal                                               
aplikacja    30.4     4.3    21.7      26.1     17.4
telefon       9.1    36.4     9.1      27.3     18.2
web          10.9    19.6    30.4      19.6     19.6


Zadanie 4e

## Wnioski biznesowe

Kategoria historyczna i obyczajowa wygenerowały największy łączny przychód,
co sugeruje że klienci preferują literaturę poważną nad gatunkową.
Dominującym kanałem sprzedaży jest web (ok. 60% zamówień) — warto inwestować w UX strony internetowej.
Warszawa i Kraków to miasta z największą liczbą zamówień, co wskazuje na koncentrację klientów w dużych miastach.
Sprzedaż jest rozłożona równomiernie między miesiącami — brak wyraźnego sezonu sprzedażowego w analizowanym okresie.